# BIST harness: A–E DEĞERLENDİRMESİ (Kaggle GPU + Ollama)
**Ayarlar (Settings):** Accelerator = **GPU T4 x2** · Internet = **On** · Persistence = Files only.
**Girdiler (+ Add Input):** (1) değerlendirme paketini içeren dataset (`eval_bundle.zip`, `python -m experiments.colab.make_eval_bundle`), (2) **eğitim notebook'unun çıktısı** (Notebook Output Files: içinde `result/*.gguf` bulunan sürüm).
**Akış:** hücre 1–5'i elle çalıştırın (şablon denetimi sonucunu görün), sonra **Save Version → Save & Run All (Commit)**: hücre 6 (A–E koşusu, T4'te tahmini 4 saat) arka planda sürer. Sonuç: sürüm sayfası → Output → `eval_v22/` (`results.csv`, `summary.json`).
Kesilebilir: koşucu her oturumdan sonra satır yazar; aynı `OUT` ile yeniden başlatınca biten satırlar atlanır. Ayrıntı: `docs/harness_design.md` §14.5. Bu not defteri Kaggle'da denenmemiştir.

In [ ]:
!nvidia-smi
import os, shutil, glob, subprocess, time, urllib.request
print({p: round(shutil.disk_usage(p).free / 1e9, 1) for p in ("/kaggle/working", "/tmp")})

In [ ]:
# Ollama kurulumu ve sunucu (yalnızca yerel 127.0.0.1; tek GPU)
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
os.environ.update(OLLAMA_HOST='127.0.0.1:11434', OLLAMA_NUM_PARALLEL='1', OLLAMA_KEEP_ALIVE='60m', CUDA_VISIBLE_DEVICES='0')
server = subprocess.Popen(['ollama', 'serve'], stdout=open('/kaggle/working/ollama.log', 'w'), stderr=subprocess.STDOUT, env=os.environ.copy())
for _ in range(90):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2)
        break
    except Exception:
        time.sleep(1)
print('Ollama hazır')

In [ ]:
# Paket: eval_bundle.zip (repo kodu + snapshot'lar + değerlendirme soruları)
import zipfile
zips = glob.glob('/kaggle/input/**/eval_bundle.zip', recursive=True)
assert zips, "Dataset ekli değil: + Add Input ile eval_bundle.zip içeren dataset'i ekleyin"
shutil.rmtree('/kaggle/working/bist', ignore_errors=True)
zipfile.ZipFile(zips[0]).extractall('/kaggle/working/bist')
os.chdir('/kaggle/working/bist')
!pip install -q 'pydantic>=2' pandas numpy pyarrow
!ls data/snapshots data/training

In [ ]:
# Modeller: temel gemma3:4b + ince ayarlı GGUF (training/ollama/Modelfile)
!ollama pull gemma3:4b
ggufs = [g for g in glob.glob('/kaggle/input/**/*.gguf', recursive=True) if 'q4_k_m' in os.path.basename(g).lower()]
assert ggufs, "İnce ayarlı GGUF bulunamadı: + Add Input ile eğitim notebook'unun çıktısını (result/*.gguf) ekleyin"
print('GGUF:', ggufs[0])
shutil.copy(ggufs[0], 'training/ollama/gemma3-bist-q4_k_m.gguf')
!cd training/ollama && ollama create gemma3-bist -f Modelfile
!ollama list

In [ ]:
# Şablon denetimi (ZORUNLU): Modelfile şablonu eğitim metniyle AYNI token sayısını vermeli (FARK varsa eğitim/çıkarım şablonu uyuşmuyor)
!HARNESS_LLM_TIMEOUT=600 python -m experiments.check_template --model gemma3-bist --records data/training/v2_2/val.jsonl --n 2

## A–E koşusu (YALNIZCA Save & Run All ile)
39 soru x 2 model x 5 profil = 390 koşu. Ardından tekrarlanabilirlik: ince ayarlı model, profil D, 20 soru, 2 tekrar.

In [ ]:
N = 40
OUT = '/kaggle/working/eval_v22'
os.environ['HARNESS_LLM_TIMEOUT'] = '900'
!python -m experiments.colab.plan --gpu T4 --n {N} --calibrate {OUT}/results.jsonl
!python -m experiments.eval_runner --questions data/training/v2_2/test.jsonl --n {N} --models gemma3:4b gemma3-bist --profiles A B C D E --out {OUT}
!python -m experiments.eval_runner --questions data/training/v2_2/test.jsonl --n 20 --models gemma3-bist --profiles D --repeats 2 --out /kaggle/working/eval_v22_repro

In [ ]:
!cat /kaggle/working/eval_v22/summary.json | head -80
!cat /kaggle/working/eval_v22_repro/summary.json | head -30